# Day 6 · Exercise 5: Prompt Library with Five Transformers

**What you'll build:** `apply_transform(transform_name: str, text: str) -> str` — a dispatcher that routes a named transform ("classify", "summarise", "extract", "fix_grammar", "rewrite_formal") to the matching prompt library builder and returns the model's response as a string.

**Why it matters:** A prompt library only pays off when there is a clean, single entry-point that callers can use without knowing which builder to call — once you have that dispatcher, any downstream code (APIs, scripts, tests) can drive the whole library through one consistent interface.

## Your Implementation

In [ ]:
import ollama

MODEL = "llama3.2"

# ---------------------------------------------------------------------------
# Prompt library — five builder functions (from Lesson 5)
# ---------------------------------------------------------------------------

_SENTIMENT_EXAMPLES: list[tuple[str, str]] = [
    ("I absolutely love this product — it changed my workflow!", "POSITIVE"),
    ("The support took three weeks to respond. Completely unacceptable.", "NEGATIVE"),
    ("The package arrived on Wednesday as expected.", "NEUTRAL"),
]

_GRAMMAR_EXAMPLES: list[tuple[str, str]] = [
    ("Fix the grammar: she go to the store", "She goes to the store."),
    ("Fix the grammar: they was happy", "They were happy."),
    ("Fix the grammar: he don't know nothing", "He doesn't know anything."),
]


def build_classifier_prompt(text: str) -> list[dict]:
    messages: list[dict] = [
        {
            "role": "system",
            "content": (
                "You are a sentiment analysis engine. "
                "Classify the input as POSITIVE, NEGATIVE, or NEUTRAL. "
                "Reply with exactly one word — no punctuation, no explanation."
            ),
        }
    ]
    for user_ex, label in _SENTIMENT_EXAMPLES:
        messages.append({"role": "user",      "content": user_ex})
        messages.append({"role": "assistant", "content": label})
    messages.append({"role": "user", "content": text})
    return messages


def build_summariser_prompt(text: str, max_words: int = 50) -> list[dict]:
    return [
        {
            "role": "system",
            "content": (
                "You are a concise technical writer. "
                "Summarise the provided text clearly and accurately."
            ),
        },
        {
            "role": "user",
            "content": (
                f"{text}\n\n"
                f"Summarise the above in at most {max_words} words. "
                "Write one paragraph. Do not add a heading or preamble."
            ),
        },
    ]


def build_extractor_prompt(text: str, fields: list[str]) -> list[dict]:
    field_list = ", ".join(f'"{f}"' for f in fields)
    return [
        {
            "role": "system",
            "content": (
                "You are a precise data extraction assistant. "
                "Extract requested fields and return them as a raw JSON object. "
                "Do not add any text before or after the JSON object."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Extract the following fields from the text below: {field_list}.\n\n"
                f"Text: {text}\n\n"
                "Return only a raw JSON object with those exact keys. "
                "If a field is not found, use null.\n"
                "Your JSON:\n{"
            ),
        },
    ]


def build_grammar_prompt(text: str) -> list[dict]:
    messages: list[dict] = [
        {
            "role": "system",
            "content": (
                "Correct the grammar of the sentence. "
                "Reply with only the corrected sentence — no explanation."
            ),
        }
    ]
    for user_ex, corrected in _GRAMMAR_EXAMPLES:
        messages.append({"role": "user",      "content": user_ex})
        messages.append({"role": "assistant", "content": corrected})
    messages.append({"role": "user", "content": f"Fix the grammar: {text}"})
    return messages


def build_rewriter_prompt(text: str, tone: str) -> list[dict]:
    return [
        {
            "role": "system",
            "content": (
                f"You are a professional editor. "
                f"Rewrite text in a {tone} tone while preserving all meaning. "
                "Do not add or remove facts. Keep the same approximate length."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Rewrite the following text in a {tone} tone:\n\n{text}\n\n"
                "Rewritten version:"
            ),
        },
    ]


# ---------------------------------------------------------------------------
# Your task: implement apply_transform below
# ---------------------------------------------------------------------------

def apply_transform(transform_name: str, text: str) -> str:
    """Dispatch a named transform to the matching prompt builder and call Ollama.

    Supported transform names and what they do:
        "classify"      — sentiment classification (POSITIVE / NEGATIVE / NEUTRAL)
        "summarise"     — condense to at most 50 words
        "extract"       — extract 'name', 'date', 'amount' as a JSON object
        "fix_grammar"   — return the grammatically corrected sentence
        "rewrite_formal" — rewrite the text in a formal tone

    Args:
        transform_name: One of the five supported transform names listed above.
        text: The input text to process.

    Returns:
        The model's response as a stripped string.

    Raises:
        ValueError: If transform_name is not one of the five supported names.

    Example:
        result = apply_transform("classify", "I love this product!")
        # result == "POSITIVE"  (or similar)
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _run_checks():
    score, total = 0, 5

    # Check 1: function exists, is callable, and returns a string on a basic call
    try:
        assert callable(apply_transform), 'apply_transform is not defined or not callable'
        result = apply_transform("classify", "test")
        assert isinstance(result, str), f'apply_transform must return str, got {type(result).__name__}'
        print(f'{_PASS} Check 1/{total}: apply_transform exists, is callable, and returns str')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: unsupported transform name raises ValueError
    try:
        raised = False
        try:
            apply_transform("nonexistent_transform", "test")
        except ValueError:
            raised = True
        assert raised, 'apply_transform("nonexistent_transform", ...) did not raise ValueError'
        print(f'{_PASS} Check 2/{total}: unsupported transform name raises ValueError')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return

    # Check 3: "classify" returns a non-empty string
    try:
        result = apply_transform("classify", "I absolutely love this product!")
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        assert len(result.strip()) > 0, 'apply_transform returned an empty string'
        print(f'{_PASS} Check 3/{total}: "classify" returns a non-empty string (got {result.strip()!r})')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: "fix_grammar" returns a non-empty string
    try:
        result = apply_transform("fix_grammar", "she go to the store every day")
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        assert len(result.strip()) > 0, 'apply_transform returned an empty string'
        print(f'{_PASS} Check 4/{total}: "fix_grammar" returns a non-empty string (got {result.strip()!r})')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # Check 5: "rewrite_formal" returns a non-empty string
    try:
        result = apply_transform("rewrite_formal", "hey just checking in — got the thing done btw")
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        assert len(result.strip()) > 0, 'apply_transform returned an empty string'
        print(f'{_PASS} Check 5/{total}: "rewrite_formal" returns a non-empty string')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {total}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

In Day 7 you will build a pipeline that chains transforms together: run "fix_grammar" first, then "summarise" the corrected text. As a preview, write a helper `def pipeline(steps: list[str], text: str) -> str` that applies each named transform in sequence, passing the output of one as the input to the next.

```python
# Example target behaviour:
result = pipeline(["fix_grammar", "summarise"], "she go to the shop and buyed many item")
print(result)  # a short summary of the grammatically corrected sentence
```

This foreshadows **chained prompting** — a core agent-building technique where the output of one model call becomes the input to the next.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def apply_transform(transform_name: str, text: str) -> str:
    """Dispatch a named transform to the matching prompt builder and call Ollama."""
    _DISPATCH: dict[str, list[dict]] = {
        "classify":       build_classifier_prompt(text),
        "summarise":      build_summariser_prompt(text, max_words=50),
        "extract":        build_extractor_prompt(text, ["name", "date", "amount"]),
        "fix_grammar":    build_grammar_prompt(text),
        "rewrite_formal": build_rewriter_prompt(text, "formal"),
    }
    if transform_name not in _DISPATCH:
        supported = ", ".join(f'"{k}"' for k in _DISPATCH)
        raise ValueError(
            f"Unknown transform {transform_name!r}. Supported: {supported}"
        )
    messages = _DISPATCH[transform_name]
    response = ollama.chat(model=MODEL, messages=messages)
    return response["message"]["content"].strip()
```

**Why this works:** the dispatch dictionary maps every supported name to a pre-built messages list — building all five is cheap (pure Python, no I/O), so it is safe to do at the top of the function. The `if` guard then catches unknown names before any model call is attempted, producing a clear `ValueError` with the full list of supported transforms. Callers get one clean interface; the prompt library handles all the structure internally. The function is plain `def` (not `async def`) because `ollama.chat()` is a synchronous blocking call — there is no async I/O here and no `await` is needed.
</details>